# Experiment 01 — Vanilla Transformer Baseline
Train a standard transformer on the same data as TMT for fair comparison.

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from tmt.data.dataset import load_text_dataset
from tmt.training.scheduler import cosine_warmup_scheduler
import math

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN = 256
BATCH_SIZE = 16

loaders = load_text_dataset('wikitext-2', seq_len=SEQ_LEN, batch_size=BATCH_SIZE)
print('Train batches:', len(loaders['train']))

In [ ]:
# Standard transformer — same param budget as TMT
baseline = nn.Transformer(
    d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
    dim_feedforward=2048, dropout=0.1, batch_first=True
).to(DEVICE)
print(f'Baseline params: {sum(p.numel() for p in baseline.parameters())/1e6:.2f}M')

In [ ]:
# Simple GPT-style decoder-only baseline using nn.TransformerDecoder
vocab_size = 50258  # gpt2 tokenizer size

class BaselineGPT(nn.Module):
    def __init__(self, vocab=vocab_size, d=512, heads=8, layers=6, seq=256):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(seq, d)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=2048, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=layers)
        self.proj = nn.Linear(d, vocab)
        self.proj.weight = self.embed.weight

    def forward(self, x):
        B, S = x.shape
        pos = torch.arange(S, device=x.device).unsqueeze(0)
        h = self.embed(x) + self.pos(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(S, device=x.device)
        h = self.transformer(h, mask=mask, is_causal=True)
        return self.proj(h)

baseline = BaselineGPT().to(DEVICE)
print(f'BaselineGPT params: {sum(p.numel() for p in baseline.parameters())/1e6:.2f}M')

In [ ]:
opt = AdamW(baseline.parameters(), lr=3e-4, weight_decay=0.1)
sched = cosine_warmup_scheduler(opt, warmup_steps=200, total_steps=2000)
baseline.train()

losses = []
for step, batch in enumerate(loaders['train']):
    if step >= 2000:
        break
    ids = batch['input_ids'].to(DEVICE)
    x, y = ids[:, :-1], ids[:, 1:]
    logits = baseline(x)
    loss = nn.functional.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(baseline.parameters(), 1.0)
    opt.step(); sched.step()
    losses.append(loss.item())
    if step % 100 == 0:
        print(f'step={step:4d} loss={loss.item():.4f}')

baseline_ppl = math.exp(sum(losses[-200:]) / 200)
print(f'\nBaseline final perplexity: {baseline_ppl:.2f}')